# Semana 23 · ¿Es estable el Sistema Solar?

Este cuaderno acompaña a la sesión de proyecto de la semana 23: ecuaciones diferenciales resueltas paso a paso, resonancias y una simulación de tres cuerpos.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. Una ecuación diferencial resuelta con Euler

$$\frac{dN}{dt} = k\,N$$

La receta de Euler: $N(t + \Delta t) = N(t) + k\,N(t)\,\Delta t$. La comparamos con la solución exacta, $N_0\,e^{kt}$, para varios tamaños de paso.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

k, N0, T = 0.2, 1000.0, 10.0          # por hora, bacterias, horas

def euler_exponencial(dt):
    t, N = [0.0], [N0]
    while t[-1] < T - 1e-12:
        N.append(N[-1] + k * N[-1] * dt)
        t.append(t[-1] + dt)
    return np.array(t), np.array(N)

tt = np.linspace(0, T, 200)
plt.plot(tt, N0 * np.exp(k * tt), "k", lw=2, label="exacta")
for dt in [2.0, 1.0, 0.25]:
    t, N = euler_exponencial(dt)
    plt.plot(t, N, "o-", ms=3, label=f"Euler, Δt = {dt} h")
plt.xlabel("horas"); plt.ylabel("bacterias"); plt.legend(); plt.grid(alpha=0.3)
plt.title("dN/dt = kN: Euler se acerca a la exponencial al reducir el paso")
plt.show()

## 2. El campo de pendientes

Una ecuación diferencial dice, en cada punto $(t, N)$, qué pendiente tiene la solución que pasa por allí. Si dibujamos un segmento corto con esa pendiente en muchos puntos, obtenemos el **campo de pendientes**: un mapa de corrientes por el que las soluciones fluyen. Lo hacemos para $dN/dt = k\,N\,(1 - N/K)$, la **ecuación logística**: una población que crece como la exponencial al principio, pero que se frena al acercarse a un máximo $K$ (por ejemplo, porque se acaba la comida).

In [ ]:
K = 5000.0
f = lambda t, N: k * N * (1 - N / K)

t_g, N_g = np.meshgrid(np.linspace(0, 40, 21), np.linspace(0, 7000, 15))
pend = f(t_g, N_g)
# segmentos de igual longitud en la gráfica: normalizamos teniendo en cuenta la escala de cada eje
esc_t, esc_N = 40, 7000
L = np.sqrt((1 / esc_t)**2 + (pend / esc_N)**2)
plt.figure(figsize=(9, 5))
plt.quiver(t_g, N_g, 1 / L, pend / L, angles="xy", headwidth=0, headlength=0, headaxislength=0, color="gray", pivot="middle")

for N_ini in [200, 1000, 6500]:
    t, N = [0.0], [N_ini]
    while t[-1] < 40:
        N.append(N[-1] + f(t[-1], N[-1]) * 0.05); t.append(t[-1] + 0.05)
    plt.plot(t, N, lw=2, label=f"N(0) = {N_ini}")
plt.xlabel("t (horas)"); plt.ylabel("N"); plt.legend(); plt.title("Campo de pendientes de la ecuación logística")
plt.show()

Todas las soluciones siguen las corrientes del campo y acaban en $N = K$: es un **equilibrio estable**. Si empiezas por encima, bajas; si empiezas por debajo, subes. En cambio, $N = 0$ es un **equilibrio inestable**: cualquier población, por pequeña que sea, se aleja de él. La pregunta de esta semana, «¿es estable el Sistema Solar?», es de este tipo, solo que con muchas más variables.

## 3. El columpio: resonancia

Un oscilador con frecuencia natural $\omega_0$ recibe un empuje periódico con frecuencia $\omega$:

$$\frac{d^2x}{dt^2} = -\omega_0^2\,x + F\cos(\omega t).$$

Una ecuación de segundo orden se resuelve con dos ecuaciones de primer orden: la posición cambia con la velocidad y la velocidad cambia con la aceleración. Usamos el método de Euler-Cromer de la fase 1 (primero se actualiza la velocidad y con la nueva, la posición), que funciona bien para oscilaciones.

In [ ]:
w0, F = 1.0, 0.05

def oscilador_forzado(w, t_max=200, dt=0.01):
    x, v = 0.0, 0.0
    t = np.arange(0, t_max, dt)
    xs = np.empty_like(t)
    for i, ti in enumerate(t):
        a = -w0**2 * x + F * np.cos(w * ti)
        v += a * dt
        x += v * dt
        xs[i] = x
    return t, xs

fig, axs = plt.subplots(3, 1, figsize=(10, 6), sharex=True, sharey=True)
for ax, w in zip(axs, [0.5, 0.9, 1.0]):
    t, x = oscilador_forzado(w)
    ax.plot(t, x, lw=0.8)
    ax.set_title(f"empuje con ω = {w} (frecuencia natural ω₀ = {w0})", fontsize=9)
axs[-1].set_xlabel("tiempo")
plt.tight_layout(); plt.show()

**Ejercicio.** Calcula la amplitud máxima alcanzada (el mayor valor de $|x|$) para frecuencias de empuje entre 0,5 y 1,5 y dibuja la amplitud frente a la frecuencia. Es una **curva de resonancia**.

In [ ]:
# Tu código aquí

In [ ]:
# Solución
ws = np.linspace(0.5, 1.5, 41)
amp = [np.abs(oscilador_forzado(w)[1]).max() for w in ws]
plt.plot(ws, amp, "o-", ms=3)
plt.xlabel("frecuencia del empuje ω"); plt.ylabel("amplitud máxima"); plt.grid(alpha=0.3)
plt.title("Curva de resonancia"); plt.show()

## 4. Tres cuerpos: el Sol, Júpiter y los asteroides

Ahora la simulación de verdad. Para que sea rápida usamos el **problema restringido de tres cuerpos**:

- El Sol y Júpiter giran en órbitas circulares alrededor de su centro de masas (lo calculamos con fórmulas, como en la semana 20).
- Los asteroides son tan pequeños que no atraen a nadie: solo sienten al Sol y a Júpiter. Así podemos simular muchos a la vez, sin que se influyan entre ellos, con las operaciones de NumPy sobre todos de golpe.

Colocamos 161 asteroides entre 2 y 3,6 ua, todos con excentricidad inicial 0,1 y en puntos al azar de sus órbitas, y los seguimos 4000 años. Unidades: ua, años y masas solares, con $G = 4\pi^2$ (semana 20).

In [ ]:
G = 4 * np.pi**2
m_J, a_J = 1 / 1047.6, 5.203
M = 1 + m_J
w_J = 2 * np.pi / np.sqrt(a_J**3 / M)        # velocidad angular de Júpiter

def sol_jupiter(t):
    """Posiciones y velocidades del Sol y de Júpiter en el instante t (órbitas circulares)."""
    u = np.array([np.cos(w_J * t), np.sin(w_J * t)])
    du = w_J * np.array([-np.sin(w_J * t), np.cos(w_J * t)])
    return -m_J / M * a_J * u, a_J / M * u, -m_J / M * a_J * du

def aceleracion(p, t):
    s, j, _ = sol_jupiter(t)
    ds, dj = s - p, j - p
    rs = np.linalg.norm(ds, axis=1)[:, None]
    rj = np.linalg.norm(dj, axis=1)[:, None]
    return G * ds / rs**3 + G * m_J * dj / rj**3

def excentricidad(p, v, t):
    """Excentricidad de la órbita de cada asteroide respecto al Sol."""
    s, _, vs = sol_jupiter(t)
    r, w = p - s, v - vs
    rn = np.linalg.norm(r, axis=1)
    a = 1 / (2 / rn - np.sum(w**2, axis=1) / G)            # vis viva, despejando a
    h = r[:, 0] * w[:, 1] - r[:, 1] * w[:, 0]              # momento angular por unidad de masa
    return np.sqrt(np.maximum(0, 1 - h**2 / (G * a)))

La función `excentricidad` usa dos cosas que conoces: la vis viva (semana 18), despejada para obtener el semieje mayor $a$, y el área barrida por unidad de tiempo (segunda ley de Kepler), que está relacionada con la excentricidad. No hace falta que entiendas cada línea; lo importante es que, a partir de la posición y la velocidad, calcula la forma de la órbita.

La simulación tarda alrededor de medio minuto. Ten paciencia.

In [ ]:
rng = np.random.default_rng(0)
a0 = np.linspace(2.0, 3.6, 161)
e0 = 0.1
fase = rng.random(a0.size) * 2 * np.pi

# Cada asteroide empieza en su perihelio, en una dirección al azar
s, _, vs = sol_jupiter(0.0)
r_p = a0 * (1 - e0)
v_p = np.sqrt(G * (2 / r_p - 1 / a0))
p = s + np.column_stack([r_p * np.cos(fase), r_p * np.sin(fase)])
v = vs + np.column_stack([-v_p * np.sin(fase), v_p * np.cos(fase)])

dt, años = 0.01, 4000
t = 0.0
acc = aceleracion(p, t)
historial_e = []
for i in range(int(años / dt)):
    v += 0.5 * dt * acc
    p += dt * v
    t += dt
    acc = aceleracion(p, t)
    v += 0.5 * dt * acc
    if i % 50 == 0:
        historial_e.append(excentricidad(p, v, t))
historial_e = np.array(historial_e)
print("Simulación terminada:", historial_e.shape[0], "medidas de excentricidad para", a0.size, "asteroides")

## 5. Dónde actúan las resonancias

Para cada asteroide medimos cuánto ha variado su excentricidad: la diferencia entre la mayor y la menor. Si Júpiter apenas le afecta, la variación es pequeña. Si los tirones se acumulan, es grande. Marcamos las posiciones de las resonancias calculadas con la tercera ley de Kepler: un asteroide en resonancia $p$:$q$ tiene un periodo $q/p$ veces el de Júpiter, así que $a = a_J\,(q/p)^{2/3}$.

In [ ]:
variacion = historial_e.max(axis=0) - historial_e.min(axis=0)

plt.figure(figsize=(10, 4.5))
plt.semilogy(a0, variacion, "o-", ms=3)
for p_, q_ in [(4, 1), (3, 1), (5, 2), (7, 3), (2, 1)]:
    a_res = a_J * (q_ / p_) ** (2 / 3)
    plt.axvline(a_res, color="r", lw=0.8, ls="--")
    plt.text(a_res + 0.01, variacion.min() * 1.1, f"{p_}:{q_}", color="r")
plt.xlabel("semieje mayor inicial (ua)"); plt.ylabel("variación de la excentricidad")
plt.title("Cuánto altera Júpiter la órbita de cada asteroide en 4000 años")
plt.grid(alpha=0.3, which="both"); plt.show()

La variación es pequeña y crece poco a poco al acercarse a Júpiter, salvo en unos picos estrechos: justo en las resonancias 3:1, 5:2 y 7:3, y en una zona ancha alrededor de la 2:1. Son los lugares de los huecos de Kirkwood. Aparece también un pico en 2,06 ua, la resonancia 4:1, que marca aproximadamente el borde interior del cinturón de asteroides. En 4000 años los cambios todavía son modestos; en millones de años, las órbitas de las resonancias se alargan hasta cruzar la de Marte, y los asteroides acaban expulsados.

**Ejercicio.** Elige el asteroide más cercano a 2,50 ua y otro a 2,30 ua y dibuja su excentricidad a lo largo del tiempo en la misma gráfica. Los tiempos de las medidas son `np.arange(len(historial_e)) * 50 * dt`.

In [ ]:
# Tu código aquí

In [ ]:
# Solución
tiempos = np.arange(len(historial_e)) * 50 * dt
for a_obj in [2.30, 2.50]:
    i = np.argmin(np.abs(a0 - a_obj))
    plt.plot(tiempos, historial_e[:, i], label=f"a = {a0[i]:.2f} ua")
plt.xlabel("años"); plt.ylabel("excentricidad"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 6. Experimenta

1. Cambia la excentricidad inicial de los asteroides a 0,2. ¿Se hacen más anchos los picos?
2. Haz que Júpiter tenga el doble de masa. ¿Qué pasa con la altura de los picos?
3. Simula solo 1000 años. ¿Se ven todas las resonancias? Las resonancias actúan lentamente: cuanto más débiles, más tiempo hace falta para que se noten.

In [ ]:
# Tu experimento aquí